In [1]:
"""
Solutions to the end-of-chapter exercises (Géron, Hands-On ML, Ch.6 — Decision Trees)

Exercise 7: Train and fine-tune a decision tree on the moons dataset.
Exercise 8: Grow a "forest" by hand — train 1,000 trees on random subsets,
            then combine their predictions via majority vote.
"""


'\nSolutions to the end-of-chapter exercises (Géron, Hands-On ML, Ch.6 — Decision Trees)\n\nExercise 7: Train and fine-tune a decision tree on the moons dataset.\nExercise 8: Grow a "forest" by hand — train 1,000 trees on random subsets,\n            then combine their predictions via majority vote.\n'

In [2]:
import numpy as np
from scipy.stats import mode
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split, GridSearchCV, ShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.base import clone


/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [3]:
# ---------------------------------------------------------------------------
# 7. Train and fine-tune a decision tree on the moons dataset
# ---------------------------------------------------------------------------
def exercise_7(random_state=42):
    # a. Generate a moons dataset using make_moons(n_samples=10000, noise=0.4)
    X, y = make_moons(n_samples=10000, noise=0.4, random_state=random_state)

    # b. Split into training and test set using train_test_split()
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state
    )

    # c. Use grid search with cross-validation (with the help of the
    #    GridSearchCV class) to find good hyperparameter values for a
    #    DecisionTreeClassifier. Hint: try various values for max_leaf_nodes.
    param_grid = {
        "max_leaf_nodes": list(range(2, 100)),
        "min_samples_split": [2, 3, 4],
    }
    tree_clf = DecisionTreeClassifier(random_state=random_state)
    grid_search = GridSearchCV(
        tree_clf, param_grid, cv=3, scoring="accuracy", n_jobs=-1, verbose=1
    )
    grid_search.fit(X_train, y_train)

    print("Ex7 - Best params:", grid_search.best_params_)
    print("Ex7 - Best CV accuracy:", grid_search.best_score_)

    # d. Train it on the full training set using these hyperparameters, and
    #    measure your model's performance on the test set. You should get
    #    roughly 85% to 87% accuracy.
    best_tree = grid_search.best_estimator_  # already refit on full X_train
    y_pred = best_tree.predict(X_test)
    test_acc = accuracy_score(y_test, y_pred)
    print("Ex7 - Test accuracy:", test_acc)
    # In the book's run: max_leaf_nodes around 17, test accuracy ~86%.
    return best_tree, test_acc


In [4]:
# ---------------------------------------------------------------------------
# 8. Grow a forest by hand
# ---------------------------------------------------------------------------
def exercise_8(best_tree_params, X_train, y_train, X_test, y_test,
               n_trees=1000, n_instances=100, random_state=42):
    """
    best_tree_params: the .get_params() dict from exercise_7's best_estimator_
                       (or just pass the fitted tree and call .get_params()).
    """
    # a. Generate 1,000 subsets of the training set, each containing 100
    #    instances selected randomly. Hint: use Scikit-Learn's ShuffleSplit
    #    for this.
    rs = ShuffleSplit(
        n_splits=n_trees,
        test_size=len(X_train) - n_instances,  # everything not in the subset
        random_state=random_state,
    )
    mini_sets = []
    for mini_train_index, mini_test_index in rs.split(X_train):
        X_mini_train = X_train[mini_train_index]
        y_mini_train = y_train[mini_train_index]
        mini_sets.append((X_mini_train, y_mini_train))

    # b. Train one decision tree on each subset, using the best
    #    hyperparameter values found in the previous exercise. Evaluate
    #    these 1,000 decision trees on the test set. Since they were
    #    trained on smaller sets, these decision trees will likely perform
    #    worse than the first decision tree, achieving only about 80% accuracy.
    forest = [DecisionTreeClassifier(**best_tree_params) for _ in range(n_trees)]

    accuracy_scores = []
    for tree, (X_mini_train, y_mini_train) in zip(forest, mini_sets):
        tree.fit(X_mini_train, y_mini_train)
        y_pred = tree.predict(X_test)
        accuracy_scores.append(accuracy_score(y_test, y_pred))

    mean_individual_acc = np.mean(accuracy_scores)
    print("Ex8b - Mean accuracy of the 1,000 individual trees:",
          mean_individual_acc)

    # c. Now comes the magic. For each test set instance, generate the
    #    predictions of the 1,000 decision trees, and keep only the most
    #    frequent prediction (you can use SciPy's mode() function for this).
    #    This approach gives you majority-vote predictions over the test set.
    Y_pred = np.empty([n_trees, len(X_test)], dtype=np.uint8)
    for tree_index, tree in enumerate(forest):
        Y_pred[tree_index] = tree.predict(X_test)

    y_pred_majority_votes, n_votes = mode(Y_pred, axis=0, keepdims=False)

    # d. Evaluate these predictions on the test set: you should obtain a
    #    slightly higher accuracy than your first model (about 0.5 to 1.5%
    #    higher). Congratulations, you have trained a random forest classifier!
    forest_acc = accuracy_score(y_test, y_pred_majority_votes)
    print("Ex8d - Majority-vote (forest) accuracy:", forest_acc)

    return forest, mean_individual_acc, forest_acc


if __name__ == "__main__":
    print("=== Exercise 7 ===")
    X, y = make_moons(n_samples=10000, noise=0.4, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    best_tree, test_acc = exercise_7()

    print("\n=== Exercise 8 ===")
    best_params = best_tree.get_params()
    exercise_8(best_params, X_train, y_train, X_test, y_test)


=== Exercise 7 ===
Fitting 3 folds for each of 294 candidates, totalling 882 fits


/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.che

Ex7 - Best params: {'max_leaf_nodes': 17, 'min_samples_split': 2}
Ex7 - Best CV accuracy: 0.8555001986342105
Ex7 - Test accuracy: 0.8695

=== Exercise 8 ===
Ex8b - Mean accuracy of the 1,000 individual trees: 0.8054865
Ex8d - Majority-vote (forest) accuracy: 0.872
